# GPT-2 Native Generation Error Profile Experiment

**Pipeline:** Generate text with pre-trained GPT-2 → GEC (coedit-large) → ERRANT error annotation → Analysis

This notebook runs the full `gen_gec_errant` pipeline on learner corpus data using
pre-trained GPT-2 (native, no fine-tuning) to establish a **native model baseline**.

**What this does:**
1. Takes learner English sentences, splits each into a prompt (first half) and reference continuation (second half)
2. Feeds prompts to GPT-2 to generate continuations
3. Runs grammatical error correction (GEC) on the generated text
4. Uses ERRANT to compare original vs. corrected text and classify error types
5. Analyzes error profiles: types, frequencies, distributions

## 0. Setup & Installation

In [ ]:
# Install the gen_gec_errant package and dependencies
!pip install -q torch transformers errant spacy numpy scipy matplotlib pandas pyyaml
!python -m spacy download en_core_web_sm -q

In [ ]:
# Clone the repository and install the package
import os

REPO_URL = "https://github.com/YOUR_USERNAME/automatic-generation-correction-errortagging-v2.git"  # <-- UPDATE THIS
REPO_DIR = "/content/gen-gec-errant"

if not os.path.exists(REPO_DIR):
    # Option A: Clone from GitHub (update URL above)
    # !git clone {REPO_URL} {REPO_DIR}
    # !pip install -e {REPO_DIR}

    # Option B: Upload the src/ directory manually, then:
    # Create a minimal package structure
    os.makedirs(REPO_DIR, exist_ok=True)
    print("Please upload or clone the gen_gec_errant package.")
    print("Either:")
    print(f"  A) Update REPO_URL above and uncomment the git clone lines")
    print(f"  B) Upload the src/ folder to {REPO_DIR}/src/ and run: pip install -e {REPO_DIR}")
else:
    print(f"Repository already present at {REPO_DIR}")
    !pip install -e {REPO_DIR} -q

In [ ]:
import logging
logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(name)s | %(message)s")

import json
import time
from pathlib import Path

import torch
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 1. Data Loading

Upload your learner corpus CSV or mount Google Drive.

Expected CSV schema: `writing_id, l1, cefr_level, text`

In [1]:
# ── Option A: Mount Google Drive ──
# from google.colab import drive
# drive.mount('/content/drive')
# DATA_PATH = "/content/drive/MyDrive/path/to/norm-CELVA-SP.csv"

# ── Option B: Upload files ──
# from google.colab import files
# uploaded = files.upload()  # Upload your CSV
# DATA_PATH = list(uploaded.keys())[0]

# ── Option C: Use built-in dummy data for quick testing ──
DUMMY_DATA = [
    "The student was trying to explain the complicated problem to his teacher but he could not find the right words to describe what happened",
    "Yesterday I went to the big shopping centre near my house and I bought many things that I did not really need at all",
    "She has been studying English for three years now and she is making very good progress with her grammar and vocabulary skills",
    "The weather was really bad last weekend so we decided to stay at home and watch movies instead of going to the park",
    "My friend told me that she wanted to travel to many different countries because she loves learning about new cultures and traditions",
    "The children were playing happily in the garden when suddenly it started raining very heavily and they all ran inside the house",
    "I think that learning a new language is one of the most difficult but rewarding things that a person can do in their life",
    "He works very hard every day at his job in the factory but he still does not earn enough money to support his family",
    "The teacher asked the students to write an essay about their favourite holiday and most of them chose to write about summer vacation",
    "We arrived at the airport very early in the morning because we did not want to miss our flight to London for the conference",
    "She told me that the most important thing in life is to be kind to other people and to always try to help those in need",
    "The book I read last month was very interesting and it taught me many things about the history of ancient civilizations and their customs",
    "I have always wanted to learn how to play the piano but I never had enough time or money to take lessons when I was younger",
    "The company decided to hire more workers because the demand for their products has been increasing rapidly over the past few months",
    "When I was a child I used to spend every summer at my grandparents house in the countryside where I would play in the fields",
    "The government announced new rules about pollution and environmental protection which will affect many factories and businesses across the country",
    "She is one of the most talented singers I have ever heard and I believe she will become very famous in the next few years",
    "The restaurant we visited last night had excellent food and the service was also very good although the prices were a bit expensive",
    "He explained that the reason he was late for the meeting was because his car broke down on the highway and he had to wait",
    "My parents always encouraged me to study hard and do my best in school because they believed education was the key to success",
]

USE_DUMMY = True  # Set to False if using real data
DATA_PATH = None  # Set this if USE_DUMMY is False

In [ ]:
# ── Configuration ──
MAX_SENTENCES = 20        # Number of sentences to process (None = all)
PROMPT_RATIO = 0.5        # Fraction of sentence used as prompt
MIN_WORDS = 10            # Minimum sentence length
MAX_WORDS = 500           # Maximum sentence length
MIN_PROMPT_WORDS = 5      # Minimum prompt length
BATCH_SIZE = 4            # Generation/GEC batch size (increase for GPU)
DEVICE = "auto"           # "auto", "cuda", or "cpu"
SEED = 42
OUTPUT_DIR = "/content/experiment/gpt2-native"

os.makedirs(OUTPUT_DIR, exist_ok=True)

In [ ]:
from gen_gec_errant.data_loader.runner import load_sentences, make_prompts

if USE_DUMMY:
    sentences = [s for s in DUMMY_DATA if MIN_WORDS <= len(s.split()) <= MAX_WORDS]
    if MAX_SENTENCES:
        sentences = sentences[:MAX_SENTENCES]
    print(f"Using {len(sentences)} dummy sentences")
else:
    sentences = load_sentences(
        path=DATA_PATH,
        max_sentences=MAX_SENTENCES,
        min_words=MIN_WORDS,
        max_words=MAX_WORDS,
        text_column="text",
    )
    print(f"Loaded {len(sentences)} sentences from {DATA_PATH}")

items = make_prompts(sentences, prompt_ratio=PROMPT_RATIO, min_prompt_words=MIN_PROMPT_WORDS)

print(f"\nPrepared {len(items)} prompt-reference pairs")
print(f"\nExample:")
print(f"  Full:      {items[0]['full']}")
print(f"  Prompt:    {items[0]['prompt']}")
print(f"  Reference: {items[0]['reference']}")

In [ ]:
# Save prompts for reproducibility
with open(Path(OUTPUT_DIR) / "prompts.json", "w") as f:
    json.dump(items, f, indent=2)
print(f"Saved prompts to {OUTPUT_DIR}/prompts.json")

## 2. Text Generation with GPT-2 Native

In [ ]:
from gen_gec_errant.generation.runner import get_device, load_model, generate_continuations, compute_perplexity
from gen_gec_errant.generation.config import ModelConfig, GenerationParams

torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = get_device(DEVICE)
print(f"Using device: {device}")

# Load pre-trained GPT-2 (native, no fine-tuning)
model_config = ModelConfig(
    name="gpt2-small",
    hf_model_id="gpt2",
    model_family="gpt2",
    is_learner_tuned=False,
)

gen_params = GenerationParams(
    max_new_tokens=50,
    min_new_tokens=10,
    temperature=1.0,
    top_k=50,
    top_p=0.95,
    do_sample=True,
    repetition_penalty=1.2,
)

print("Loading GPT-2...")
model, tokenizer = load_model(model_config, device)
print(f"Model loaded: {sum(p.numel() for p in model.parameters()) / 1e6:.1f}M parameters")

In [ ]:
prompts = [item["prompt"] for item in items]

print(f"Generating continuations for {len(prompts)} prompts...")
t0 = time.time()

continuations = generate_continuations(
    model, tokenizer, prompts, gen_params,
    batch_size=BATCH_SIZE, device=device,
)

full_texts = [f"{p} {c}" for p, c in zip(prompts, continuations)]

print(f"Computing perplexity...")
perplexities = compute_perplexity(
    model, tokenizer, full_texts,
    batch_size=BATCH_SIZE, device=device,
)

elapsed = time.time() - t0
print(f"\nGeneration complete in {elapsed:.1f}s")
print(f"Mean perplexity: {np.mean(perplexities):.2f}")

# Free GPU memory
del model
if device.type == "cuda":
    torch.cuda.empty_cache()

In [ ]:
# Show some examples
print("=" * 80)
for i in range(min(5, len(items))):
    print(f"\n--- Sentence {i+1} (PPL: {perplexities[i]:.2f}) ---")
    print(f"  Prompt:        {prompts[i]}")
    print(f"  GPT-2 cont:    {continuations[i]}")
    print(f"  Learner ref:   {items[i]['reference']}")

In [ ]:
# Build results dict
all_results = {
    "gpt2-small": {
        "continuations": continuations,
        "full_texts": full_texts,
        "perplexities": perplexities,
        "prompt_boundaries": [len(p) for p in prompts],
    },
    # Add learner baseline for comparison
    "learner_baseline": {
        "continuations": [item["reference"] for item in items],
        "full_texts": [item["full"] for item in items],
        "perplexities": [0.0] * len(items),
        "prompt_boundaries": [len(item["prompt"]) for item in items],
    },
}
print(f"Results prepared for {len(all_results)} models: {list(all_results.keys())}")

## 3. Grammatical Error Correction (GEC)

In [ ]:
from gen_gec_errant.gec.runner import run_gec
from gen_gec_errant.gec.config import GECConfig

gec_config = GECConfig(
    method="dedicated",
    model_id="grammarly/coedit-large",
    batch_size=BATCH_SIZE,
    device=DEVICE,
)

print("Running GEC with coedit-large...")
t0 = time.time()

for model_name, results in all_results.items():
    print(f"  Correcting {model_name} outputs...")
    run_gec(gec_config, results)

elapsed = time.time() - t0
print(f"\nGEC complete in {elapsed:.1f}s")

In [ ]:
# Show GEC examples
print("=" * 80)
print("GEC Correction Examples (GPT-2 continuations):")
for i in range(min(5, len(items))):
    orig = all_results["gpt2-small"]["continuations"][i]
    corr = all_results["gpt2-small"]["corrected_continuations"][i]
    changed = "CHANGED" if orig != corr else "no change"
    print(f"\n--- Sentence {i+1} [{changed}] ---")
    print(f"  Original:  {orig}")
    print(f"  Corrected: {corr}")

## 4. ERRANT Error Annotation

In [ ]:
from gen_gec_errant.annotation.runner import run_annotation, classify_errors_by_region, summarize_errors_by_region
from gen_gec_errant.annotation.config import AnnotationConfig

ann_config = AnnotationConfig(lang="en")

print("Running ERRANT annotation...")
t0 = time.time()

for model_name, results in all_results.items():
    print(f"  Annotating {model_name}...")
    run_annotation(ann_config, results)

    # Region classification on full-text annotations
    if "full_text_annotations" in results and "prompt_boundaries" in results:
        classify_errors_by_region(results["full_text_annotations"], results["prompt_boundaries"])
        results["region_error_summary"] = summarize_errors_by_region(results["full_text_annotations"])

elapsed = time.time() - t0
print(f"\nAnnotation complete in {elapsed:.1f}s")

In [ ]:
# Show error annotation examples
print("=" * 80)
print("Error Annotation Examples (GPT-2 continuations):")
gpt2_anns = all_results["gpt2-small"]["annotations"]
for i in range(min(5, len(gpt2_anns))):
    ann = gpt2_anns[i]
    print(f"\n--- Sentence {i+1}: {ann.num_errors} error(s) ---")
    print(f"  Original:  {ann.original}")
    print(f"  Corrected: {ann.corrected}")
    for err in ann.errors:
        print(f"    [{err.error_type}] '{err.original_tokens}' -> '{err.corrected_tokens}'")

## 5. Analysis & Results

In [ ]:
from gen_gec_errant.analysis.runner import run_analysis
from gen_gec_errant.analysis.config import AnalysisConfig

analysis_config = AnalysisConfig(
    output_dir=OUTPUT_DIR,
    skip_plots=False,
    top_n_error_types=10,
)

# Save raw results first
def serialize_annotations(annotations):
    return [
        {
            "original": a.original,
            "corrected": a.corrected,
            "num_errors": a.num_errors,
            "error_types": a.error_type_counts,
            "prompt_error_count": a.prompt_error_count,
            "generation_error_count": a.generation_error_count,
            "errors": [
                {
                    "orig_tokens": e.original_tokens,
                    "corr_tokens": e.corrected_tokens,
                    "type": e.error_type,
                    "char_start": e.char_start,
                    "char_end": e.char_end,
                    "region": e.region,
                }
                for e in a.errors
            ],
        }
        for a in annotations
    ]

raw_output = {}
for model_name, results in all_results.items():
    raw_output[model_name] = {
        "continuations": results["continuations"],
        "full_texts": results["full_texts"],
        "corrected_continuations": results.get("corrected_continuations", []),
        "corrected_full_texts": results.get("corrected_full_texts", []),
        "perplexities": results["perplexities"],
        "error_summary": results.get("error_summary", {}),
        "annotations": serialize_annotations(results.get("annotations", [])),
        "full_text_annotations": serialize_annotations(results.get("full_text_annotations", [])),
        "region_error_summary": results.get("region_error_summary", {}),
    }

with open(Path(OUTPUT_DIR) / "raw_results.json", "w") as f:
    json.dump(raw_output, f, indent=2, default=str)
print(f"Saved raw results to {OUTPUT_DIR}/raw_results.json")

# Run analysis
summaries, comparison = run_analysis(analysis_config, all_results, items)
print("\nAnalysis complete!")

In [ ]:
# ── Summary Table ──
print("=" * 90)
print("MODEL COMPARISON")
print("=" * 90)
print(f"{'Model':<20} {'PPL Mean':>10} {'PPL Std':>10} {'Err Rate':>10} {'Avg Err':>10} {'Total Err':>10}")
print("-" * 90)
for s in summaries:
    print(
        f"{s['model_name']:<20} "
        f"{s['ppl_mean']:>10.2f} "
        f"{s['ppl_std']:>10.2f} "
        f"{s['error_rate']:>10.3f} "
        f"{s['avg_errors_per_sentence']:>10.3f} "
        f"{s['total_errors']:>10d}"
    )
print("=" * 90)

In [ ]:
# ── Top Error Types ──
for s in summaries:
    print(f"\n{'='*50}")
    print(f"Top 10 Error Types: {s['model_name']}")
    print(f"{'='*50}")
    for rank, (etype, count) in enumerate(s['top_10_error_types'], 1):
        bar = '#' * min(count, 40)
        print(f"  {rank:>2}. {etype:<15} {count:>4}  {bar}")

## 6. Visualizations

In [ ]:
# ── Perplexity Distribution ──
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart: mean perplexity
model_names = [s['model_name'] for s in summaries if s['model_name'] != 'learner_baseline']
ppl_means = [s['ppl_mean'] for s in summaries if s['model_name'] != 'learner_baseline']
ppl_stds = [s['ppl_std'] for s in summaries if s['model_name'] != 'learner_baseline']

axes[0].bar(model_names, ppl_means, yerr=ppl_stds, capsize=5, color='steelblue', alpha=0.8)
axes[0].set_ylabel('Perplexity')
axes[0].set_title('Mean Perplexity by Model')
axes[0].tick_params(axis='x', rotation=30)

# Histogram: perplexity distribution
gpt2_ppls = all_results['gpt2-small']['perplexities']
axes[1].hist(gpt2_ppls, bins=20, color='steelblue', alpha=0.7, edgecolor='white')
axes[1].axvline(np.mean(gpt2_ppls), color='red', linestyle='--', label=f'Mean: {np.mean(gpt2_ppls):.1f}')
axes[1].axvline(np.median(gpt2_ppls), color='orange', linestyle='--', label=f'Median: {np.median(gpt2_ppls):.1f}')
axes[1].set_xlabel('Perplexity')
axes[1].set_ylabel('Count')
axes[1].set_title('GPT-2 Perplexity Distribution')
axes[1].legend()

plt.tight_layout()
plt.savefig(Path(OUTPUT_DIR) / 'plots' / 'perplexity_overview.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Error Comparison: GPT-2 vs Learner Baseline ──
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

names = [s['model_name'] for s in summaries]
err_rates = [s['error_rate'] for s in summaries]
avg_errs = [s['avg_errors_per_sentence'] for s in summaries]
colors = ['steelblue' if n != 'learner_baseline' else 'coral' for n in names]

axes[0].bar(names, err_rates, color=colors, alpha=0.8)
axes[0].set_ylabel('Error Rate')
axes[0].set_title('Error Rate by Model')
axes[0].tick_params(axis='x', rotation=30)

axes[1].bar(names, avg_errs, color=colors, alpha=0.8)
axes[1].set_ylabel('Avg Errors per Sentence')
axes[1].set_title('Average Errors per Sentence')
axes[1].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.savefig(Path(OUTPUT_DIR) / 'plots' / 'error_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Error Type Breakdown (side-by-side) ──
fig, axes = plt.subplots(1, len(summaries), figsize=(7 * len(summaries), 6))
if len(summaries) == 1:
    axes = [axes]

for ax, s in zip(axes, summaries):
    top_types = s['top_10_error_types'][:10]
    if not top_types:
        ax.set_title(f"{s['model_name']}\n(no errors)")
        continue
    types, counts = zip(*top_types)
    color = 'coral' if s['model_name'] == 'learner_baseline' else 'steelblue'
    ax.barh(list(reversed(types)), list(reversed(counts)), color=color, alpha=0.8)
    ax.set_xlabel('Count')
    ax.set_title(f"{s['model_name']}\nTop Error Types")

plt.tight_layout()
plt.savefig(Path(OUTPUT_DIR) / 'plots' / 'error_type_breakdown.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── PPL vs Errors Scatter ──
gpt2_summary = next(s for s in summaries if s['model_name'] == 'gpt2-small')
ppls = [x['ppl'] for x in gpt2_summary['per_sentence_ppl_plus_errors']]
errs = [x['errors'] for x in gpt2_summary['per_sentence_ppl_plus_errors']]

fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(ppls, errs, alpha=0.6, color='steelblue', edgecolors='white', s=60)
ax.set_xlabel('Perplexity')
ax.set_ylabel('Number of Errors')
ax.set_title('GPT-2 Native: Perplexity vs. Error Count')

# Add trend line
if len(ppls) > 2:
    z = np.polyfit(ppls, errs, 1)
    p = np.poly1d(z)
    x_line = np.linspace(min(ppls), max(ppls), 100)
    ax.plot(x_line, p(x_line), 'r--', alpha=0.5, label=f'Trend (slope={z[0]:.3f})')
    ax.legend()

plt.tight_layout()
plt.savefig(Path(OUTPUT_DIR) / 'plots' / 'ppl_vs_errors.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Exported Files

In [ ]:
# List all output files
print("Generated output files:")
for p in sorted(Path(OUTPUT_DIR).rglob("*")):
    if p.is_file():
        size_kb = p.stat().st_size / 1024
        print(f"  {p.relative_to(OUTPUT_DIR)}  ({size_kb:.1f} KB)")

In [ ]:
# ── Preview: full_results.tsv ──
tsv_path = Path(OUTPUT_DIR) / "full_results.tsv"
if tsv_path.exists():
    df = pd.read_csv(tsv_path, sep="\t")
    print(f"full_results.tsv: {df.shape[0]} rows x {df.shape[1]} columns")
    print(f"\nColumns: {list(df.columns)}")
    display(df.head())
else:
    print("full_results.tsv not found")

In [ ]:
# ── Preview: errors_long_format.tsv ──
err_path = Path(OUTPUT_DIR) / "errors_long_format.tsv"
if err_path.exists():
    df_err = pd.read_csv(err_path, sep="\t")
    print(f"errors_long_format.tsv: {df_err.shape[0]} rows x {df_err.shape[1]} columns")
    if len(df_err) > 0:
        print(f"\nError type distribution:")
        display(df_err['error_type'].value_counts().head(15))
        print(f"\nSample errors:")
        display(df_err[['model', 'error_type', 'error_original_tokens', 'error_corrected_tokens', 'region']].head(10))
else:
    print("errors_long_format.tsv not found")

In [ ]:
# ── Download results (Colab) ──
# Uncomment to download results as a zip file:
# import shutil
# shutil.make_archive('/content/gpt2_native_results', 'zip', OUTPUT_DIR)
# from google.colab import files
# files.download('/content/gpt2_native_results.zip')

## 8. Summary

This notebook ran the full **generate -> GEC -> ERRANT -> analysis** pipeline:

1. **Data**: Learner English sentences split into prompts + reference continuations
2. **Generation**: Pre-trained GPT-2 generated continuations from prompts
3. **GEC**: `grammarly/coedit-large` corrected grammatical errors
4. **ERRANT**: Classified errors by type (M:DET, R:VERB:SVA, U:PRON, etc.)
5. **Analysis**: Computed error profiles, compared GPT-2 vs. learner baseline

**Key outputs:**
- `raw_results.json` — Complete pipeline data
- `full_results.tsv` — Flat CSV (1 row per sentence)
- `errors_long_format.tsv` — Long format (1 row per error) for R/statistical analysis
- `*_summary.json` — Per-model metrics
- `plots/` — Visualizations